# Planificador basico de recursos Kubernetes

Este notebook estima consumo total de CPU y memoria a partir de `requests` declarados. Sirve como utilidad sencilla para practicar capacity planning antes de desplegar varias workloads.


In [ ]:
def parse_cpu(value):
    value = str(value).strip()
    if value.endswith("m"):
        return int(value[:-1]) / 1000
    return float(value)


def parse_memory_mib(value):
    value = str(value).strip()
    if value.endswith("Gi"):
        return float(value[:-2]) * 1024
    if value.endswith("Mi"):
        return float(value[:-2])
    raise ValueError(f"Formato no soportado: {value}")


def summarize_workloads(workloads):
    total_cpu = 0.0
    total_mem = 0.0
    for workload in workloads:
        replicas = workload["replicas"]
        cpu = parse_cpu(workload["cpu_request"])
        mem = parse_memory_mib(workload["memory_request"])
        total_cpu += replicas * cpu
        total_mem += replicas * mem
    return {
        "cpu_cores": round(total_cpu, 3),
        "memory_mib": round(total_mem, 1),
        "memory_gib": round(total_mem / 1024, 2),
    }


def estimate_nodes(total_cpu_cores, total_memory_mib, node_cpu_cores=2, node_memory_mib=4096):
    import math

    cpu_nodes = math.ceil(total_cpu_cores / node_cpu_cores)
    mem_nodes = math.ceil(total_memory_mib / node_memory_mib)
    return max(cpu_nodes, mem_nodes)


In [ ]:
workloads = [
    {"name": "api", "replicas": 3, "cpu_request": "200m", "memory_request": "256Mi"},
    {"name": "worker", "replicas": 2, "cpu_request": "300m", "memory_request": "512Mi"},
    {"name": "web", "replicas": 2, "cpu_request": "100m", "memory_request": "128Mi"},
]

summary = summarize_workloads(workloads)
estimated_nodes = estimate_nodes(summary["cpu_cores"], summary["memory_mib"])

print("Resumen total:", summary)
print("Nodos estimados (2 CPU / 4 Gi por nodo):", estimated_nodes)


## Siguientes mejoras

- Incorporar `limits` y comparar sobreasignacion.
- Leer datos desde un CSV o una hoja de calculo.
- Estimar costes aproximados por tamano de cluster.
